<a href="https://colab.research.google.com/github/fware/FaceSwap/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Training Demo
This is a simple example for training the SimSwap 224*224 with VGGFace2-224.

Code path: https://github.com/neuralchen/SimSwap
If you like the SimSwap project, please star it!
Paper path: https://arxiv.org/pdf/2106.06340v1.pdf or https://dl.acm.org/doi/10.1145/3394171.3413630

In [1]:
!nvidia-smi

Sun May 10 23:51:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

Installation
All file changes made by this notebook are temporary. You can try to mount your own google drive to store files if you want.

#Get Scripts

In [2]:
!git clone https://github.com/neuralchen/SimSwap
!cd SimSwap && git pull

Cloning into 'SimSwap'...
remote: Enumerating objects: 1141, done.
remote: Counting objects: 100% (508/508), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 1141 (delta 419), reused 382 (delta 382), pack-reused 633 (from 1)
Receiving objects: 100% (1141/1141), 211.46 MiB | 41.20 MiB/s, done.
Resolving deltas: 100% (601/601), done.
Already up to date.


# Install Blocks

In [3]:
!pip install googledrivedownloader
!pip install timm==0.5.4
!wget -P SimSwap/arcface_model https://github.com/neuralchen/SimSwap/releases/download/1.0/arcface_checkpoint.tar

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 21.2 MB/s eta 0:00:00
  Attempting uninstall: timm
    Found existing installation: timm 1.0.26
    Uninstalling timm-1.0.26:
      Successfully uninstalled timm-1.0.26
--2026-05-10 23:51:37--  https://github.com/neuralchen/SimSwap/releases/download/1.0/arcface_checkpoint.tar
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/374891081/f01468b3-446b-4867-8c78-6d496183f9e6?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-11T00%3A36%3A20Z&rscd=attachment%3B+filename%3Darcface_checkpoint.tar&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-10T23%3A35%3A35Z&ske=2026-05-11T00%3A36%3A20Z&sks=b&skv=2018-11-09&sig=hWvEVmvK5VN7cCpH0wsOXgwYpqhO9rE%2BUdV4TDj%2B

#Download the Training Dataset
We employ the cropped VGGFace2-224 dataset for this toy training demo.

You can download the dataset from our google driver https://drive.google.com/file/d/19pWvdEHS-CEG6tW3PdxdtZ5QEymVjImc/view?usp=sharing

***Please check the dataset in dir /content/TrainingData***

***If dataset already exists in /content/TrainingData, please do not run blow scripts!***


In [ ]:
# %mkdir /content/TrainingData
# !wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=19pWvdEHS-CEG6tW3PdxdtZ5QEymVjImc' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=19pWvdEHS-CEG6tW3PdxdtZ5QEymVjImc" -O /content/TrainingData/vggface2_crop_arcfacealign_224.tar && rm -rf /tmp/cookies.txt
# %cd /content/
# !tar -xzvf /content/TrainingData/vggface2_crop_arcfacealign_224.tar -C /content/TrainingData
#  !rm /content/TrainingData/vggface2_crop_arcfacealign_224.tar

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
%cd /content/drive/MyDrive/FaceSwapML

/content/drive/MyDrive/FaceSwapML


In [6]:
!tar -xzf lfw_dataset.tar.gz -C /content/

In [9]:
!ls /content/

drive  lfw_funneled  lfw_funneled_aligned  sample_data	SimSwap


In [8]:
%mkdir /content/lfw_funneled_aligned

In [10]:
%cd /content/SimSwap/insightface_func/models/

/content/SimSwap/insightface_func/models


In [16]:
!ls

antelopev2.zip


In [17]:
import os

# Check if the file exists before attempting to unzip
if os.path.exists('antelopev2.zip'):
    !unzip antelopev2.zip
    # List the contents of the current directory to show the unzipped files
    !ls
else:
    print("antelopev2.zip not found in the current directory.")

Archive:  antelopev2.zip
   creating: antelopev2/
  inflating: antelopev2/genderage.onnx  
  inflating: antelopev2/2d106det.onnx  
  inflating: antelopev2/1k3d68.onnx  
  inflating: antelopev2/glintr100.onnx  
  inflating: antelopev2/scrfd_10g_bnkps.onnx  
antelopev2  antelopev2.zip


In [21]:
!ls antelopev2/

1k3d68.onnx    genderage.onnx  scrfd_10g_bnkps.onnx
2d106det.onnx  glintr100.onnx


In [27]:
!unzip checkpoints.zip

Archive:  checkpoints.zip
   creating: people/
  inflating: people/iter.txt         
  inflating: people/latest_net_D1.pth  
  inflating: people/latest_net_D2.pth  
  inflating: people/latest_net_G.pth  
  inflating: people/loss_log.txt     
  inflating: people/opt.txt          
   creating: people/web/
   creating: people/web/images/


In [32]:
%cd /content/SimSwap/insightface_func/
!python detect_and_align_faces.py

/content/SimSwap/insightface_func
Load InsightFace antelopev2 ONNX models
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /content/SimSwap/insightface_func/models/antelopev2/1k3d68.onnx landmark_3d_68
Applied provider

In [28]:
!pip install insightface onnxruntime-gpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 17.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 196.2 MB/s eta 0:00:00
  Created wheel for insightface: filename=insightface-0.7.3-cp312-cp312-linux_x86_64.whl size=1071490 sha256=d3e6c9ccce789c34cb757c2814aaab8cf78049aea03cf800337dd6001dab5a78
  Stored in directory: /root/.cache/pip/wheels/73/3c/e2/6d4815e8a8b33a2006554d65ce0d1f973e768f4c7a222fa675
Successfully built insightface


In [35]:
!unzip /content/SimSwap/checkpoints/checkpoints.zip

Archive:  /content/SimSwap/checkpoints/checkpoints.zip
   creating: people/
  inflating: people/iter.txt         
  inflating: people/latest_net_D1.pth  
  inflating: people/latest_net_D2.pth  
  inflating: people/latest_net_G.pth  
  inflating: people/loss_log.txt     
  inflating: people/opt.txt          
   creating: people/web/
   creating: people/web/images/


In [36]:
!ls

people	simswap224_test


In [37]:
!nvidia-smi

Mon May 11 00:45:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [44]:
%mv /content/drive/MyDrive/FaceSwapML/SimSwapModels/people /content/drive/MyDrive/FaceSwapML/SimSwapModels/simswap224_test/

#Training
Batch size must larger than 1!

In [ ]:
# %cd /content/SimSwap
# !ls
# !python train.py --name simswap224_test --gpu_ids 0 --dataset /content/TrainingData/vggface2_crop_arcfacealign_224 --Gdeep False

In [45]:
%cd /content/SimSwap
!ls
!python train.py \
  --name people \
  --which_epoch latest \
  --continue_train True \
  --dataset /content/lfw_funneled_aligned \
  --checkpoints_dir /content/drive/MyDrive/FaceSwapML/SimSwapModels \
  --gpu_ids 0 \
  --Gdeep False \
  --total_step 12000 \
  --batchSize 64

/content/SimSwap
 arcface_model	       predict.py
 checkpoints	       README.md
 cog.yaml	      'SimSwap colab.ipynb'
 crop_224	       simswaplogo
 data		       test_one_image.py
 demo_file	       test_video_swapmulti.py
 docs		       test_video_swap_multispecific.py
 download-weights.sh   test_video_swapsingle.py
 insightface_func      test_video_swapspecific.py
 LICENSE	       test_wholeimage_swapmulti.py
 models		       test_wholeimage_swap_multispecific.py
 MultiSpecific.ipynb   test_wholeimage_swapsingle.py
 options	       test_wholeimage_swapspecific.py
 output		       train.ipynb
 parsing_model	       train.py
 pg_modules	       util
2026-05-11 01:01:57.582917: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-11 01:01:57.614805: I tensorflow/core/platform/cp

In [46]:
%rm -r /content/drive/MyDrive/FaceSwapML/SimSwapModels/people

In [47]:
%cd /content/drive/MyDrive/FaceSwapML/SimSwapModels

/content/drive/MyDrive/FaceSwapML/SimSwapModels


In [48]:
!unzip /content/SimSwap/checkpoints/checkpoints.zip

Archive:  /content/SimSwap/checkpoints/checkpoints.zip
   creating: people/
  inflating: people/iter.txt         
  inflating: people/latest_net_D1.pth  
  inflating: people/latest_net_D2.pth  
  inflating: people/latest_net_G.pth  
  inflating: people/loss_log.txt     
  inflating: people/opt.txt          
   creating: people/web/
   creating: people/web/images/


In [49]:
%cd /content/drive/MyDrive/FaceSwapML/SimSwapModels/checkpoints/simswap224_test/

/content/drive/MyDrive/FaceSwapML/SimSwapModels/checkpoints/simswap224_test


In [51]:
!ls /content/drive/MyDrive/FaceSwapML/SimSwapModels/people

iter.txt	   latest_net_D2.pth  loss_log.txt  web
latest_net_D1.pth  latest_net_G.pth   opt.txt


In [ ]:
%cd /content/SimSwap
!ls
!python train.py \
  --name people \
  --which_epoch latest \
  --continue_train True \
  --dataset /content/lfw_funneled_aligned \
  --checkpoints_dir /content/drive/MyDrive/FaceSwapML/SimSwapModels \
  --load_pretrain /content/drive/MyDrive/FaceSwapML/SimSwapModels/people \
  --gpu_ids 0 \
  --Gdeep False \
  --total_step 100000 \
  --batchSize 64

/content/SimSwap
 arcface_model	       predict.py
 checkpoints	       README.md
 cog.yaml	      'SimSwap colab.ipynb'
 crop_224	       simswaplogo
 data		       test_one_image.py
 demo_file	       test_video_swapmulti.py
 docs		       test_video_swap_multispecific.py
 download-weights.sh   test_video_swapsingle.py
 insightface_func      test_video_swapspecific.py
 LICENSE	       test_wholeimage_swapmulti.py
 models		       test_wholeimage_swap_multispecific.py
 MultiSpecific.ipynb   test_wholeimage_swapsingle.py
 options	       test_wholeimage_swapspecific.py
 output		       train.ipynb
 parsing_model	       train.py
 pg_modules	       util
2026-05-11 01:41:10.476049: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-11 01:41:10.508068: I tensorflow/core/platform/cp